# bias_correction notebook

This notebook embeds the full code from `bias_correction.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Bias-correct EURO-CORDEX scenario forcing using the HISTORICAL experiment as reference.

Methodology (matches thesis "Future scenario" section):
- Reference period: 1976-2005 (default; CMIP5/CORDEX standard 30-year window, fully inside the historical run).
- Spatial extraction (default: areal): area-weighted mean over EUR-11 grid cells intersecting the eStreams
  catchment polygon (overlap fraction in EPSG:3035). Fallback: nearest cell to basin lat/lon (--extraction nearest).
- Variables: pr, tas, tasmin, tasmax.
- Correction factors are computed per day-of-year (DOY) using a centred 30-day moving window (+/-15 days)
  pooled across all reference years, instead of one factor per calendar month. This avoids the step changes
  at month boundaries and better resolves rapid seasonal transitions (e.g. snowmelt onset, autumn cooling).
- Precipitation: MULTIPLICATIVE correction (sum of obs / sum of mod on wet days inside the window).
- Temperatures (tas, tasmin, tasmax): ADDITIVE correction (mean of daily obs - mod inside the window).
- Factors are computed from CORDEX historical vs eStreams observations, then applied to the CORDEX scenario
  series, so the climate-change signal of the scenario run is preserved.
- Future PET is computed with the Hargreaves equation using the bias-corrected tasmin, tasmax, tas.

Run examples (PowerShell, from project folder):
python bias_correction.py --basin_id SE000197
python bias_correction.py --basin_id SE000197 --extraction areal
python bias_correction.py --basin_id SE000197 --extraction nearest
python bias_correction.py --basin_id SE000197 --ref_start 1976-01-01 --ref_end 2005-12-31
python bias_correction.py --basin_id SE000197 --cordex_base "F:\\Data\\EUROCORDEX_MPI"

Expected directory layout under --cordex_base (default: F:\\Data\\EUROCORDEX_MPI):
    historical/pr/*.nc
    historical/tas/*.nc
    historical/tasmin/*.nc
    historical/tasmax/*.nc
    future/pr/*.nc
    future/tas/*.nc
    future/tasmin/*.nc
    future/tasmax/*.nc
"""

import argparse
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import NamedTuple

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pandas as pd
import pyet

try:
    import geopandas as gpd
    from shapely.geometry import Polygon
    from shapely.ops import unary_union

    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False


ROOT = Path(r"C:\Users\eiv001\Desktop\thesis\Cursor")
CORDEX_BASE_DEFAULT = Path(r"F:\Data\EUROCORDEX_MPI")
OBS_DIR_DEFAULT = ROOT / "data" / "csv_estream"
BASINS_CSV = ROOT / "data" / "csv_estream" / "basins.csv"
CATCHMENT_SHP_DEFAULT = Path(
    r"F:\Data\estreams_2025\shapefiles\shapefiles\estreams_catchments.shp"
)
OUT_DIR_DEFAULT = ROOT / "Results" / "eurocordex_bias_corrected"
# Equal-area CRS for overlap weights (LAEA Europe; valid for EUR-11 domain).
WEIGHT_CRS = "EPSG:3035"


class GridWeight(NamedTuple):
    i_lat: int
    i_lon: int
    weight: float

# Per-variable spec:
#   corr        : "mult" (multiplicative) or "add" (additive)
#   obs_col     : column name in estreams_meteorology_{basin}.csv used as observation
#   scale       : multiplier applied to raw CORDEX values (e.g. 86400 to go from kg m-2 s-1 to mm/day)
#   offset      : added after scaling (e.g. -273.15 K -> C)
#   candidates  : ordered list of acceptable netCDF variable names (first match in each file is used)
#   ylabel      : label used in the diagnostic plot
VARIABLES = {
    "pr":     {"corr": "mult", "obs_col": "p_mean", "scale": 86400.0, "offset": 0.0,
               "candidates": ["pr"],          "ylabel": "Precipitation (mm/day)"},
    "tas":    {"corr": "add",  "obs_col": "t_mean", "scale": 1.0,     "offset": -273.15,
               "candidates": ["tas", "ts"],   "ylabel": "T mean (C)"},
    "tasmin": {"corr": "add",  "obs_col": "t_min",  "scale": 1.0,     "offset": -273.15,
               "candidates": ["tasmin"],      "ylabel": "T min (C)"},
    "tasmax": {"corr": "add",  "obs_col": "t_max",  "scale": 1.0,     "offset": -273.15,
               "candidates": ["tasmax"],      "ylabel": "T max (C)"},
}


# ----------------------- NetCDF helpers (point extraction) -----------------------

def to_datetime_index(time_var) -> pd.DatetimeIndex:
    """Parse NetCDF time to midnight dates (drops hour offset, e.g. 12:00 from CORDEX)."""
    dt = nc.num2date(time_var[:], units=time_var.units, calendar=getattr(time_var, "calendar", "standard"))
    return pd.to_datetime([x.strftime("%Y-%m-%d") for x in dt])


def nearest_ij(lat2d: np.ndarray, lon2d: np.ndarray, lat0: float, lon0: float) -> tuple[int, int]:
    dlat = lat2d - lat0
    dlon = (lon2d - lon0) * np.cos(np.deg2rad(lat0))
    return np.unravel_index(np.nanargmin(dlat ** 2 + dlon ** 2), lat2d.shape)


def read_lat_lon(ds: nc.Dataset) -> tuple[np.ndarray, np.ndarray, str, str]:
    lat_name = "latitude" if "latitude" in ds.variables else "lat"
    lon_name = "longitude" if "longitude" in ds.variables else "lon"
    lat = np.array(ds.variables[lat_name][:], dtype=float)
    lon = np.array(ds.variables[lon_name][:], dtype=float)
    if lat.ndim == 1 and lon.ndim == 1:
        lon2d, lat2d = np.meshgrid(lon, lat)
        return lat2d, lon2d, lat_name, lon_name
    return lat, lon, lat_name, lon_name


def make_index_for_var(var, time_name: str, lat_name: str, lon_name: str,
                       ds: nc.Dataset, i_lat: int, i_lon: int):
    lat_dims = tuple(ds.variables[lat_name].dimensions)
    lon_dims = tuple(ds.variables[lon_name].dimensions)
    shared = tuple(d for d in lat_dims if d in lon_dims)
    idx = []
    for dim in var.dimensions:
        if dim == time_name:
            idx.append(slice(None))
        elif dim == lat_name or (len(shared) == 2 and dim == shared[0]) or (len(lat_dims) == 1 and dim == lat_dims[0]):
            idx.append(i_lat)
        elif dim == lon_name or (len(shared) == 2 and dim == shared[1]) or (len(lon_dims) == 1 and dim == lon_dims[0]):
            idx.append(i_lon)
        else:
            idx.append(0)
    return tuple(idx)


def read_point_series(nc_files: list[Path], var_candidates: list[str], lat0: float, lon0: float,
                      scale: float = 1.0, offset: float = 0.0) -> pd.Series:
    """Extract a daily time series at the cell nearest to (lat0, lon0).

    var_candidates: ordered list of acceptable netCDF variable names. In each file the first
    candidate that is present is used (e.g. tas / ts).
    """
    parts: list[pd.Series] = []
    ij = None
    for f in nc_files:
        with nc.Dataset(f) as ds:
            current_var = next((v for v in var_candidates if v in ds.variables), None)
            if current_var is None:
                continue
            time_name = "time"
            lat2d, lon2d, lat_name, lon_name = read_lat_lon(ds)
            if ij is None:
                ij = nearest_ij(lat2d, lon2d, lat0, lon0)
            var = ds.variables[current_var]
            values = np.array(
                var[make_index_for_var(var, time_name, lat_name, lon_name, ds, ij[0], ij[1])],
                dtype=float,
            ).reshape(-1)
            if np.ma.isMaskedArray(values):
                values = np.ma.filled(values, np.nan)
            dates = to_datetime_index(ds.variables[time_name])
            parts.append(pd.Series(values * scale + offset, index=dates))
    if not parts:
        raise FileNotFoundError(f"No NetCDF file contained any of {var_candidates}")
    out = pd.concat(parts).sort_index()
    return out[~out.index.duplicated(keep="first")]


# ----------------------- Areal extraction (catchment-weighted) -----------------------

def load_catchment_geometry(basin_id: str, shp_path: Path):
    """Load one eStreams catchment polygon (EPSG:4326)."""
    if not HAS_GEOPANDAS:
        raise ImportError("geopandas and shapely are required for --extraction areal")
    if not shp_path.exists():
        raise FileNotFoundError(f"Catchment shapefile not found: {shp_path}")
    try:
        gdf = gpd.read_file(shp_path, where=f"basin_id = '{basin_id}'")
    except Exception:
        gdf = gpd.read_file(shp_path)
        gdf = gdf.loc[gdf["basin_id"].astype(str) == basin_id]
    if gdf.empty:
        raise KeyError(f"basin_id {basin_id!r} not found in {shp_path}")
    geom = gdf.geometry.iloc[0]
    if geom.geom_type == "MultiPolygon":
        geom = unary_union(geom)
    if not geom.is_valid:
        geom = geom.buffer(0)
    return geom


def read_grid_vertices(ds: nc.Dataset) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, str]:
    """Return 2D lat/lon centres plus optional lat_vertices/lon_vertices (nlat, nlon, 4)."""
    lat2d, lon2d, lat_name, lon_name = read_lat_lon(ds)
    if "lat_vertices" in ds.variables and "lon_vertices" in ds.variables:
        latv = np.array(ds.variables["lat_vertices"][:], dtype=float)
        lonv = np.array(ds.variables["lon_vertices"][:], dtype=float)
        return lat2d, lon2d, latv, lonv, lat_name, lon_name
    return lat2d, lon2d, None, None, lat_name, lon_name


def _cell_polygon(latv: np.ndarray, lonv: np.ndarray, i: int, j: int) -> Polygon:
    coords = [(float(lonv[i, j, k]), float(latv[i, j, k])) for k in range(latv.shape[2])]
    poly = Polygon(coords)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly


def compute_areal_weights(
    catchment,
    lat_vertices: np.ndarray,
    lon_vertices: np.ndarray,
    *,
    bbox_buffer_deg: float = 0.3,
    weight_crs: str = WEIGHT_CRS,
) -> list[GridWeight]:
    """Fraction of catchment area overlapping each EUR-11 grid cell (sums to 1)."""
    if not HAS_GEOPANDAS:
        raise ImportError("geopandas and shapely are required for areal extraction")

    catch_gdf = gpd.GeoDataFrame(geometry=[catchment], crs="EPSG:4326")
    catch_proj = catch_gdf.to_crs(weight_crs)
    catch_area = float(catch_proj.geometry.area.iloc[0])
    if catch_area <= 0:
        raise ValueError("Catchment area is zero after projection.")

    minx, miny, maxx, maxy = catchment.bounds
    minx -= bbox_buffer_deg
    maxx += bbox_buffer_deg
    miny -= bbox_buffer_deg
    maxy += bbox_buffer_deg

    lat_c = lat_vertices.mean(axis=2)
    lon_c = lon_vertices.mean(axis=2)
    candidates = np.argwhere(
        (lon_c >= minx) & (lon_c <= maxx) & (lat_c >= miny) & (lat_c <= maxy)
    )

    # EStreams-style weights: w_i = area(cell_i ∩ catchment) / Σ area(cell ∩ catchment)
    weights: list[GridWeight] = []
    catch_geom_proj = catch_proj.geometry.iloc[0]
    for i, j in candidates:
        cell = _cell_polygon(lat_vertices, lon_vertices, int(i), int(j))
        if not cell.intersects(catchment):
            continue
        cell_proj = gpd.GeoDataFrame(geometry=[cell], crs="EPSG:4326").to_crs(weight_crs)
        overlap = float(cell_proj.intersection(catch_geom_proj).area.iloc[0])
        if overlap <= 0:
            continue
        weights.append(GridWeight(int(i), int(j), overlap))

    if not weights:
        raise ValueError("No CORDEX grid cells overlap the catchment polygon.")
    total = sum(w.weight for w in weights)
    return [GridWeight(w.i_lat, w.i_lon, w.weight / total) for w in weights]


def weights_cache_path(out_dir: Path, basin_id: str) -> Path:
    return out_dir / f"grid_weights_{basin_id}.csv"


def save_grid_weights(path: Path, weights: list[GridWeight], extraction: str, basin_id: str) -> None:
    df = pd.DataFrame(
        {
            "basin_id": basin_id,
            "extraction": extraction,
            "rlat_idx": [w.i_lat for w in weights],
            "rlon_idx": [w.i_lon for w in weights],
            "weight": [w.weight for w in weights],
        }
    )
    df.to_csv(path, index=False)


def load_grid_weights(path: Path) -> list[GridWeight]:
    df = pd.read_csv(path)
    required = {"rlat_idx", "rlon_idx", "weight"}
    if not required.issubset(df.columns):
        raise ValueError(f"Invalid weights file {path}; need columns {required}")
    return [
        GridWeight(int(r.rlat_idx), int(r.rlon_idx), float(r.weight))
        for r in df.itertuples(index=False)
    ]


def get_grid_weights(
    basin_id: str,
    template_nc: Path,
    catchment_shp: Path,
    out_dir: Path,
    *,
    force_recompute: bool,
    bbox_buffer_deg: float,
) -> tuple[list[GridWeight], str, str, str]:
    """Load cached weights or compute from template NetCDF + catchment polygon."""
    cache = weights_cache_path(out_dir, basin_id)
    if cache.exists() and not force_recompute:
        weights = load_grid_weights(cache)
        print(f"  Loaded {len(weights)} grid weights from {cache.name} (sum={sum(w.weight for w in weights):.4f})")
        with nc.Dataset(template_nc) as ds:
            _, _, _, _, lat_name, lon_name = read_grid_vertices(ds)
        return weights, lat_name, lon_name, "areal"

    catchment = load_catchment_geometry(basin_id, catchment_shp)
    with nc.Dataset(template_nc) as ds:
        _, _, latv, lonv, lat_name, lon_name = read_grid_vertices(ds)
    if latv is None or lonv is None:
        raise ValueError(
            f"{template_nc.name} has no lat_vertices/lon_vertices; use --extraction nearest "
            "or a EURO-CORDEX file with cell corner coordinates."
        )
    weights = compute_areal_weights(
        catchment, latv, lonv, bbox_buffer_deg=bbox_buffer_deg,
    )
    save_grid_weights(cache, weights, "areal", basin_id)
    print(
        f"  Areal extraction: {len(weights)} cells overlap catchment "
        f"(weights -> {cache.name})"
    )
    return weights, lat_name, lon_name, "areal"


def read_areal_series_loop(
    nc_files: list[Path],
    weights: list[GridWeight],
    var_candidates: list[str],
    lat_name: str,
    lon_name: str,
    scale: float = 1.0,
    offset: float = 0.0,
) -> pd.Series:
    """Area-weighted series; one NetCDF read per overlapping cell (slow, low memory)."""
    parts: list[pd.Series] = []
    for f in nc_files:
        with nc.Dataset(f) as ds:
            current_var = next((v for v in var_candidates if v in ds.variables), None)
            if current_var is None:
                continue
            time_name = "time"
            var = ds.variables[current_var]
            n_time = len(ds.variables[time_name])
            acc = np.zeros(n_time, dtype=float)
            for w in weights:
                vals = np.array(var[:, w.i_lat, w.i_lon], dtype=float).reshape(-1)
                if np.ma.isMaskedArray(vals):
                    vals = np.ma.filled(vals, np.nan)
                acc += w.weight * vals
            dates = to_datetime_index(ds.variables[time_name])
            parts.append(pd.Series(acc * scale + offset, index=dates))
    if not parts:
        raise FileNotFoundError(f"No NetCDF file contained any of {var_candidates}")
    out = pd.concat(parts).sort_index()
    return out[~out.index.duplicated(keep="first")]


def read_areal_series(
    nc_files: list[Path],
    weights: list[GridWeight],
    var_candidates: list[str],
    lat_name: str,
    lon_name: str,
    scale: float = 1.0,
    offset: float = 0.0,
) -> pd.Series:
    """Area-weighted daily series (EStreams-style bulk read + weighted sum per file)."""
    ii = np.array([w.i_lat for w in weights], dtype=np.intp)
    jj = np.array([w.i_lon for w in weights], dtype=np.intp)
    ww = np.array([w.weight for w in weights], dtype=float)
    parts: list[pd.Series] = []
    for f in nc_files:
        with nc.Dataset(f) as ds:
            current_var = next((v for v in var_candidates if v in ds.variables), None)
            if current_var is None:
                continue
            time_name = "time"
            var = ds.variables[current_var]
            data = np.asarray(var[:], dtype=float)
            if np.ma.isMaskedArray(data):
                data = np.ma.filled(data, np.nan)
            block = data[:, ii, jj]
            acc = np.nansum(block * ww, axis=1)
            dates = to_datetime_index(ds.variables[time_name])
            parts.append(pd.Series(acc * scale + offset, index=dates))
    if not parts:
        raise FileNotFoundError(f"No NetCDF file contained any of {var_candidates}")
    out = pd.concat(parts).sort_index()
    return out[~out.index.duplicated(keep="first")]


def extract_series(
    nc_files: list[Path],
    var_candidates: list[str],
    *,
    extraction: str,
    lat0: float,
    lon0: float,
    weights: list[GridWeight] | None,
    lat_name: str | None,
    lon_name: str | None,
    scale: float,
    offset: float,
) -> pd.Series:
    if extraction == "nearest":
        return read_point_series(nc_files, var_candidates, lat0, lon0, scale=scale, offset=offset)
    if weights is None or lat_name is None or lon_name is None:
        raise ValueError("Areal extraction requires precomputed grid weights.")
    return read_areal_series(
        nc_files, weights, var_candidates, lat_name, lon_name, scale=scale, offset=offset,
    )


# ----------------------- Bias-correction core (30-day DOY window) -----------------------

def moving_window_factors(model: pd.Series, obs: pd.Series, ref_start: str, ref_end: str,
                          half_window: int, corr: str) -> pd.Series:
    """Compute one correction value per day-of-year (1..366) over the reference period.

    For each target DOY d, all reference-period days whose DOY falls inside [d - half_window,
    d + half_window] (modulo 366) are pooled. Within this pool:
    - corr == "mult": factor = sum(obs) / sum(mod) on days with mod > 0.01 mm/day
      (avoids median(obs/mod) collapsing to 0 when many dry days have obs=0)
    - corr == "add" : factor = mean (obs - mod)   (temperature)
    """
    m = model.loc[ref_start:ref_end]
    o = obs.loc[ref_start:ref_end]
    df = pd.concat([o.rename("obs"), m.rename("mod")], axis=1).dropna()
    if df.empty:
        raise ValueError(
            f"No overlap between observations and historical model over {ref_start}..{ref_end}."
        )
    doys = df.index.dayofyear.to_numpy()
    obs_arr = df["obs"].to_numpy()
    mod_arr = df["mod"].to_numpy()

    factors = np.full(366, np.nan)
    for d in range(1, 367):
        window_doys = ((np.arange(d - half_window, d + half_window + 1) - 1) % 366) + 1
        mask = np.isin(doys, window_doys)
        if not np.any(mask):
            continue
        if corr == "mult":
            o_w = obs_arr[mask]
            m_w = mod_arr[mask]
            wet = (m_w > 0.01) & np.isfinite(o_w) & np.isfinite(m_w)
            if np.sum(wet) >= 5:
                factors[d - 1] = float(np.sum(o_w[wet]) / np.sum(m_w[wet]))
            else:
                factors[d - 1] = 1.0
        else:
            factors[d - 1] = float(np.nanmean(obs_arr[mask] - mod_arr[mask]))

    out = pd.Series(factors, index=np.arange(1, 367))
    if corr == "mult":
        out = out.replace([np.inf, -np.inf], np.nan).fillna(1.0)
    else:
        out = out.fillna(0.0)
    return out


def apply_factors(series: pd.Series, factors_by_doy: pd.Series, corr: str) -> pd.Series:
    doy = pd.Series(series.index.dayofyear, index=series.index)
    f = doy.map(factors_by_doy).astype(float)
    out = series * f if corr == "mult" else series + f
    if corr == "mult":
        out = out.clip(lower=0.0)
    return out


def enforce_temperature_consistency(
    tmin: pd.Series,
    tmax: pd.Series,
    tmean: pd.Series | None = None,
) -> tuple[pd.Series, pd.Series, pd.Series | None]:
    """Ensure tasmin <= tasmax and tas lies between them (independent additive BC can invert min/max)."""
    lo = np.minimum(tmin.to_numpy(dtype=float), tmax.to_numpy(dtype=float))
    hi = np.maximum(tmin.to_numpy(dtype=float), tmax.to_numpy(dtype=float))
    tmin_out = pd.Series(lo, index=tmin.index)
    tmax_out = pd.Series(hi, index=tmax.index)
    if tmean is None:
        return tmin_out, tmax_out, None
    tmean_out = pd.Series(np.clip(tmean.to_numpy(dtype=float), lo, hi), index=tmean.index)
    return tmin_out, tmax_out, tmean_out


def write_bc_temperature_csv(out_dir: Path, basin_id: str, var: str, raw: pd.Series, corrected: pd.Series) -> None:
    out_csv = out_dir / f"{var}_bc_daily_{basin_id}.csv"
    pd.DataFrame({
        "date": corrected.index,
        f"{var}_raw": raw.reindex(corrected.index).values,
        f"{var}_bc": corrected.values,
    }).to_csv(out_csv, index=False)


def repair_temperature_and_pet(out_dir: Path, basin_id: str, lat_deg: float) -> int:
    """Fix tasmin/tasmax ordering and recompute PET from existing BC temperature CSVs."""
    paths = {v: out_dir / f"{v}_bc_daily_{basin_id}.csv" for v in ("tas", "tasmin", "tasmax")}
    for v, p in paths.items():
        if not p.exists():
            raise FileNotFoundError(f"Missing {p}; run full bias correction first.")

    raw_series: dict[str, pd.Series] = {}
    bc_series: dict[str, pd.Series] = {}
    for var, path in paths.items():
        df = pd.read_csv(path)
        df["date"] = pd.to_datetime(df["date"]).dt.normalize()
        df = df.set_index("date")
        raw_series[var] = pd.to_numeric(df[f"{var}_raw"], errors="coerce")
        bc_series[var] = pd.to_numeric(df[f"{var}_bc"], errors="coerce")

    n_inverted = int((bc_series["tasmin"] > bc_series["tasmax"]).sum())
    tmin, tmax, tmean = enforce_temperature_consistency(
        bc_series["tasmin"], bc_series["tasmax"], bc_series["tas"],
    )
    for var, corrected in (("tasmin", tmin), ("tasmax", tmax), ("tas", tmean)):
        write_bc_temperature_csv(out_dir, basin_id, var, raw_series[var], corrected)

    common = tmean.index.intersection(tmin.index).intersection(tmax.index)
    pet = pyet.hargreaves(
        tmin=tmin.loc[common],
        tmax=tmax.loc[common],
        tmean=tmean.loc[common],
        lat=np.deg2rad(lat_deg),
    )
    pet_csv = out_dir / f"pet_future_bc_{basin_id}.csv"
    pd.DataFrame({"date": common, "pet_mm_day": np.asarray(pet)}).to_csv(pet_csv, index=False)
    n_pet_nan = int(pd.isna(pet).sum())
    print(
        f"  Temperature repair: {n_inverted} day(s) had tasmin > tasmax; "
        f"PET recomputed -> {pet_csv.name} ({n_pet_nan} NaN remaining)"
    )
    return n_inverted


# ----------------------- Diagnostic plot -----------------------

def diagnostic_plot(dates, raw, corrected, title: str, ylabel: str, out_png: Path) -> None:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(dates, raw,       color="#c95f5f", linewidth=0.8, alpha=0.7, label="Raw CORDEX (scenario)")
    ax.plot(dates, corrected, color="#1f77b4", linewidth=0.9,            label="Bias-corrected")
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


# ----------------------- CLI / main -----------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=__doc__,
        formatter_class=argparse.RawTextHelpFormatter,
    )
    parser.add_argument("--basin_id", default="SE000197")
    parser.add_argument(
        "--basins",
        default="",
        help="Comma-separated basin IDs (batch mode). Overrides --basin_id when set.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=1,
        help="Parallel basins in batch mode (default 1). Try 2–3 on external drives.",
    )
    parser.add_argument(
        "--skip_existing",
        action="store_true",
        help="Skip basin if pr/tas/pet_future_bc outputs already exist.",
    )
    parser.add_argument("--cordex_base", default=str(CORDEX_BASE_DEFAULT))
    parser.add_argument("--obs_dir", default=str(OBS_DIR_DEFAULT))
    parser.add_argument("--out_dir", default=str(OUT_DIR_DEFAULT))
    parser.add_argument("--ref_start", default="1976-01-01")
    parser.add_argument("--ref_end", default="2005-12-31")
    parser.add_argument("--window_days", type=int, default=30,
                        help="Total moving-window width in days (centred). 30 means +/-15 days.")
    parser.add_argument("--variables", default="pr,tas,tasmin,tasmax",
                        help="Comma list of variables to bias-correct.")
    parser.add_argument(
        "--repair_temperature_pet_only",
        action="store_true",
        help="Only enforce tasmin<=tasmax and recompute PET from existing BC CSVs (no NetCDF).",
    )
    parser.add_argument(
        "--extraction",
        choices=("areal", "nearest"),
        default="areal",
        help="Spatial aggregation: areal (catchment-weighted cells) or nearest (single cell).",
    )
    parser.add_argument(
        "--catchment_shp",
        default=str(CATCHMENT_SHP_DEFAULT),
        help="eStreams catchments shapefile (basin_id column). Used for --extraction areal.",
    )
    parser.add_argument(
        "--bbox_buffer_deg",
        type=float,
        default=0.3,
        help="Lon/lat buffer around catchment bounds when searching overlapping cells.",
    )
    parser.add_argument(
        "--force_recompute_weights",
        action="store_true",
        help="Recompute and overwrite cached grid_weights_{basin_id}.csv.",
    )
    parser.add_argument(
        "--benchmark",
        action="store_true",
        help="Time nearest vs areal NetCDF extraction per variable (no bias-corrected outputs).",
    )
    return parser.parse_known_args()[0]


def run_extraction_benchmark(
    basin_id: str,
    cordex_base: Path,
    out_dir: Path,
    catchment_shp: Path,
    variables: list[str],
    *,
    lat0: float,
    lon0: float,
    force_recompute_weights: bool,
    bbox_buffer_deg: float,
) -> None:
    """Compare wall-clock time: nearest cell vs areal (bulk) extraction."""
    print(f"\n=== Extraction benchmark: {basin_id} ===\n")
    template_var = "pr" if "pr" in variables else variables[0]
    template_nc = sorted((cordex_base / "historical" / template_var).glob("*.nc"))
    if not template_nc:
        raise FileNotFoundError(f"No historical NetCDF for {template_var}")

    t0 = time.perf_counter()
    weights, lat_name, lon_name, _ = get_grid_weights(
        basin_id,
        template_nc[0],
        catchment_shp,
        out_dir,
        force_recompute=force_recompute_weights,
        bbox_buffer_deg=bbox_buffer_deg,
    )
    t_weights = time.perf_counter() - t0
    print(f"Grid weights: {len(weights)} cells, {t_weights:.1f} s\n")
    print(f"{'variable':<8} {'n_files':>7}  {'nearest_s':>10}  {'areal_bulk_s':>12}  {'ratio':>6}")
    print("-" * 58)

    totals = {"nearest": 0.0, "areal_bulk": 0.0}
    n_files_total = 0
    for var in variables:
        spec = VARIABLES[var]
        hist_files = sorted((cordex_base / "historical" / var).glob("*.nc"))
        fut_files = sorted((cordex_base / "future" / var).glob("*.nc"))
        files = hist_files + fut_files
        if not files:
            print(f"{var:<8}  (no files)")
            continue

        t0 = time.perf_counter()
        read_point_series(files, spec["candidates"], lat0, lon0, scale=spec["scale"], offset=spec["offset"])
        t_near = time.perf_counter() - t0

        t0 = time.perf_counter()
        read_areal_series(
            files, weights, spec["candidates"], lat_name, lon_name,
            scale=spec["scale"], offset=spec["offset"],
        )
        t_bulk = time.perf_counter() - t0

        ratio = t_bulk / t_near if t_near > 0 else float("nan")
        print(f"{var:<8} {len(files):>7}  {t_near:>10.1f}  {t_bulk:>12.1f}  {ratio:>5.1f}x")
        totals["nearest"] += t_near
        totals["areal_bulk"] += t_bulk
        n_files_total += len(files)

    print("-" * 58)
    print(
        f"{'TOTAL':<8} {n_files_total:>7}  {totals['nearest']:>10.1f}  "
        f"{totals['areal_bulk']:>12.1f}  "
        f"{totals['areal_bulk'] / max(totals['nearest'], 0.1):>5.1f}x"
    )
    est_bc_near = totals["nearest"] + 60
    est_bc_areal = totals["areal_bulk"] + t_weights + 60
    print(f"\nRough full bias_correction estimate (+~60 s factors/plots/PET):")
    print(f"  nearest:    ~{est_bc_near / 60:.1f} min")
    print(f"  areal bulk: ~{est_bc_areal / 60:.1f} min  (EStreams-style read per file)")


def _basin_outputs_complete(out_dir: Path, basin_id: str) -> bool:
    return all(
        (out_dir / name).exists()
        for name in (
            f"pr_bc_daily_{basin_id}.csv",
            f"tas_bc_daily_{basin_id}.csv",
            f"pet_future_bc_{basin_id}.csv",
        )
    )


def run_bias_correction_for_basin(
    basin_id: str,
    *,
    cordex_base: Path,
    obs_dir: Path,
    out_dir: Path,
    catchment_shp: Path,
    requested: list[str],
    ref_start: str,
    ref_end: str,
    half_window: int,
    extraction: str,
    force_recompute_weights: bool,
    bbox_buffer_deg: float,
    skip_existing: bool,
) -> dict:
    """Bias-correct one basin; returns {basin_id, status, ...}."""
    out_dir.mkdir(parents=True, exist_ok=True)
    if skip_existing and _basin_outputs_complete(out_dir, basin_id):
        return {"basin_id": basin_id, "status": "skipped", "reason": "outputs exist"}

    basins_df = pd.read_csv(BASINS_CSV)
    row = basins_df.loc[basins_df["basin_id"] == basin_id]
    if row.empty:
        return {"basin_id": basin_id, "status": "failed", "reason": "basin_id not in basins.csv"}
    basin = row.iloc[0]
    lat0 = float(basin["lat"])
    lon0 = float(basin["lon"])
    print(f"[{basin_id}] lat={lat0:.4f}, lon={lon0:.4f}, extraction={extraction}", flush=True)

    grid_weights: list[GridWeight] | None = None
    lat_name: str | None = None
    lon_name: str | None = None
    use_extraction = extraction
    if use_extraction == "areal":
        template_var = "pr" if "pr" in requested else requested[0]
        template_nc = sorted((cordex_base / "historical" / template_var).glob("*.nc"))
        if not template_nc:
            return {
                "basin_id": basin_id,
                "status": "failed",
                "reason": f"No historical NetCDF for {template_var}",
            }
        try:
            grid_weights, lat_name, lon_name, use_extraction = get_grid_weights(
                basin_id,
                template_nc[0],
                catchment_shp,
                out_dir,
                force_recompute=force_recompute_weights,
                bbox_buffer_deg=bbox_buffer_deg,
            )
        except Exception as exc:
            print(f"[{basin_id}] areal failed ({exc}); using nearest", flush=True)
            use_extraction = "nearest"
            grid_weights = None

    obs_csv = obs_dir / f"estreams_meteorology_{basin_id}.csv"
    if not obs_csv.exists():
        return {"basin_id": basin_id, "status": "failed", "reason": f"missing {obs_csv}"}
    obs = pd.read_csv(obs_csv)
    obs["date"] = pd.to_datetime(obs["date"]).dt.normalize()
    obs = obs.set_index("date")

    corrected: dict[str, pd.Series] = {}
    for var in requested:
        spec = VARIABLES[var]
        hist_files = sorted((cordex_base / "historical" / var).glob("*.nc"))
        fut_files = sorted((cordex_base / "future" / var).glob("*.nc"))
        if not hist_files or not fut_files:
            print(f"[{basin_id}] skip {var}: hist={len(hist_files)} fut={len(fut_files)}", flush=True)
            continue

        print(f"[{basin_id}] {var}: {len(hist_files)}+{len(fut_files)} NC files", flush=True)
        hist = extract_series(
            hist_files, spec["candidates"],
            extraction=use_extraction, lat0=lat0, lon0=lon0,
            weights=grid_weights, lat_name=lat_name, lon_name=lon_name,
            scale=spec["scale"], offset=spec["offset"],
        )
        fut = extract_series(
            fut_files, spec["candidates"],
            extraction=use_extraction, lat0=lat0, lon0=lon0,
            weights=grid_weights, lat_name=lat_name, lon_name=lon_name,
            scale=spec["scale"], offset=spec["offset"],
        )
        obs_series = pd.to_numeric(obs[spec["obs_col"]], errors="coerce")
        factors = moving_window_factors(hist, obs_series, ref_start, ref_end, half_window, spec["corr"])
        fut_bc = apply_factors(fut, factors, spec["corr"])
        corrected[var] = fut_bc

        factors.rename("factor").to_csv(
            out_dir / f"factors_{var}_{basin_id}.csv", header=True, index_label="doy",
        )
        pd.DataFrame({
            "date": fut_bc.index,
            f"{var}_raw": fut.reindex(fut_bc.index).values,
            f"{var}_bc": fut_bc.values,
        }).to_csv(out_dir / f"{var}_bc_daily_{basin_id}.csv", index=False)

        diagnostic_plot(
            dates=fut_bc.index,
            raw=fut.reindex(fut_bc.index).values,
            corrected=fut_bc.values,
            title=f"{var} - raw vs bias-corrected (scenario) - {basin_id}",
            ylabel=spec["ylabel"],
            out_png=out_dir / f"{var}_bc_daily_{basin_id}.png",
        )

    if all(k in corrected for k in ("tas", "tasmin", "tasmax")):
        repair_temperature_and_pet(out_dir, basin_id, lat0)
    else:
        print(f"[{basin_id}] PET skipped (need tas, tasmin, tasmax)", flush=True)

    return {"basin_id": basin_id, "status": "ok", "extraction": use_extraction}


def _bias_correction_worker(task: dict) -> dict:
    try:
        return run_bias_correction_for_basin(
            task["basin_id"],
            cordex_base=Path(task["cordex_base"]),
            obs_dir=Path(task["obs_dir"]),
            out_dir=Path(task["out_dir"]),
            catchment_shp=Path(task["catchment_shp"]),
            requested=task["requested"],
            ref_start=task["ref_start"],
            ref_end=task["ref_end"],
            half_window=task["half_window"],
            extraction=task["extraction"],
            force_recompute_weights=task["force_recompute_weights"],
            bbox_buffer_deg=task["bbox_buffer_deg"],
            skip_existing=task["skip_existing"],
        )
    except Exception as exc:
        return {"basin_id": task["basin_id"], "status": "failed", "reason": str(exc)}


def main() -> None:
    args = parse_args()
    cordex_base = Path(args.cordex_base)
    obs_dir = Path(args.obs_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    requested = [v.strip() for v in args.variables.split(",") if v.strip() in VARIABLES]
    if not requested:
        raise ValueError(f"No valid variables in --variables. Allowed: {list(VARIABLES.keys())}")

    half_window = max(1, int(args.window_days) // 2)
    basin_ids = [b.strip() for b in args.basins.split(",") if b.strip()] or [args.basin_id]

    if args.repair_temperature_pet_only:
        basins_df = pd.read_csv(BASINS_CSV)
        for basin_id in basin_ids:
            row = basins_df.loc[basins_df["basin_id"] == basin_id].iloc[0]
            repair_temperature_and_pet(out_dir, basin_id, float(row["lat"]))
        print("Done.")
        return

    if args.benchmark and len(basin_ids) == 1:
        basins_df = pd.read_csv(BASINS_CSV)
        basin = basins_df.loc[basins_df["basin_id"] == basin_ids[0]].iloc[0]
        run_extraction_benchmark(
            basin_ids[0],
            cordex_base,
            out_dir,
            Path(args.catchment_shp),
            requested,
            lat0=float(basin["lat"]),
            lon0=float(basin["lon"]),
            force_recompute_weights=args.force_recompute_weights,
            bbox_buffer_deg=args.bbox_buffer_deg,
        )
        return

    print(f"Basins ({len(basin_ids)}): {', '.join(basin_ids)}")
    print(f"Reference period: {args.ref_start} to {args.ref_end} (window = {args.window_days} days)")
    print(f"Spatial extraction: {args.extraction}")

    task_template = {
        "cordex_base": str(cordex_base),
        "obs_dir": str(obs_dir),
        "out_dir": str(out_dir),
        "catchment_shp": str(args.catchment_shp),
        "requested": requested,
        "ref_start": args.ref_start,
        "ref_end": args.ref_end,
        "half_window": half_window,
        "extraction": args.extraction,
        "force_recompute_weights": args.force_recompute_weights,
        "bbox_buffer_deg": args.bbox_buffer_deg,
        "skip_existing": args.skip_existing,
    }

    workers = max(1, int(args.workers))
    if workers > 1 and len(basin_ids) > 1:
        tasks = [{**task_template, "basin_id": bid} for bid in basin_ids]
        print(f"Running {len(tasks)} basins with {workers} parallel workers ...")
        results: list[dict] = []
        with ProcessPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_bias_correction_worker, t): t["basin_id"] for t in tasks}
            for fut in as_completed(futures):
                bid = futures[fut]
                res = fut.result()
                results.append(res)
                print(f"  finished {bid}: {res.get('status')} {res.get('reason', '')}", flush=True)
    else:
        results = []
        for basin_id in basin_ids:
            results.append(
                run_bias_correction_for_basin(
                    basin_id,
                    cordex_base=cordex_base,
                    obs_dir=obs_dir,
                    out_dir=out_dir,
                    catchment_shp=Path(args.catchment_shp),
                    requested=requested,
                    ref_start=args.ref_start,
                    ref_end=args.ref_end,
                    half_window=half_window,
                    extraction=args.extraction,
                    force_recompute_weights=args.force_recompute_weights,
                    bbox_buffer_deg=args.bbox_buffer_deg,
                    skip_existing=args.skip_existing,
                )
            )

    failed = [r for r in results if r.get("status") == "failed"]
    if failed:
        print(f"Failed basins ({len(failed)}):")
        for r in failed:
            print(f"  {r['basin_id']}: {r.get('reason', '')}")
    print("Done.")


# Execute the script entry point
main()
